# Synthetic Data (Neighbor Only Features) - All Explanation Methods Comparison

This notebook runs all three explanation methods (CF-Greedy, CF, CFF) on synthetic graph data where labels depend only on neighbor features.

In [ ]:
# Bootstrap: run this notebook from either the repo root or notebooks/.
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

In [ ]:
# Setup
import sys
for module in ['gen_graph_data', 'model', 'intervention_design_model', 'experiment_runner']:
    if module in sys.modules:
        del sys.modules[module]

import numpy as np
import torch
import matplotlib.pyplot as plt
from data.gen_graph_data import generate_graph,generate_graph_withAvgDegree

In [ ]:
# Generate synthetic graph with neighbor-only features
data, G, attributes, w, cutoff_score = generate_graph_withAvgDegree(
    n_nodes=250,
    m_attrs=6,     # D6: the Neighbor-Only family in Table 1 / Table 3
    alpha=5.0,
    beta=0.0,
    total_edges=300,
    label_percent=0.17,
    sbm_fraction=0.6,
    n_comms=6,
    p_intra=0.08,
    p_inter=0.006,
    seed=42,
    keep_features=False  # Labels depend only on neighbor features
)

feature_names = [f"F{i}" for i in range(data.x.shape[1])]

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}, Features: {data.x.shape[1]}")
print(f"Positive labels: {data.y.sum().item()}")

In [ ]:
# Run all three methods
from src.experiment_runner import run_ALL

results = run_ALL(
    data=data,
    feature_names=feature_names,
    N=1,
    budget=10.0,
    epsilon=0.1,
    output_file='results/test_results/synthetic_neighbor_only_ALL_results.json',
    val_ratio=0,
    test_ratio=0.1,
    hidden_dim=16,
    dropout=0.2,
    lr=0.05,
    max_epochs=3000,
    patience=50,
    check_every=300,
    explanation_method="all",
    cff_mode='feature',
    verbose=False,
    parallel=True,
    num_workers=8
)

In [ ]:
# Plot comparison
from src.utils import plot_coverage_comparison

plot_coverage_comparison(
    results_file='results/test_results/synthetic_neighbor_only_ALL_results.json',
    title='Synthetic (Neighbor Only): Coverage vs Cost',
    save_path='results/test_results/synthetic_neighbor_only_ALL_comparison.png'
)